In [ ]:
# ==============================================================================
# SCRIPT: ANÁLISE INDIVIDUAL POR CÉLULA (MAPA + SÉRIES + STL)
# ==============================================================================

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.dates as mdates
from shapely.strtree import STRtree
from statsmodels.tsa.seasonal import seasonal_decompose

# ==============================
# CONFIGURAÇÃO
# ==============================
TARGET_VAR = 'dV'   # 'dV' (Vertical) ou 'dH' (Horizontal)
GRID_SIZE = 50      # Aumente isto se tiver muitas células (ex: 100, 150)

# ==============================
# 0. Ler COS e definir barragem
# ==============================
COS_PATH = r"C:\projetos\analise_insar_ist\data\COS2023v1-S2-shp\COS2023v1-S2.shp"
try:
    cos = gpd.read_file(COS_PATH).to_crs(epsg=3035)
    barragem = cos[cos['COS23_n4_L'] == 'Infraestruturas de produção de energia hídrica']
except Exception as e:
    print(f"Aviso: Não foi possível ler o ficheiro COS. Criando placeholder. Erro: {e}")
    barragem = gpd.GeoDataFrame({'COS23_n4_L': ['Placeholder'], 'geometry': [box(2792250, 1855050, 2793250, 1855850)]}, crs="EPSG:3035")

# ==============================
# 1. Ler CSVs
# ==============================
# Certifique-se que estes ficheiros correspondem à zona da barragem escolhida acima

# Alqueva
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"

norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

#Castelo do Bode
#asc_file = "data/castelo_bode_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0233_IW1_VV_2019_2023_1/EGMS_L2b_147_0233_IW1_VV_2019_2023_1.csv"
#desc_file = "data/castelo_bode_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0841_IW3_VV_2019_2023_1/EGMS_L2b_052_0841_IW3_VV_2019_2023_1.csv"

#norte_min, norte_max = 2017250, 2018550
#este_min, este_max = 2754250, 2754850

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)

def filter_area(df): return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & (df['easting'] >= este_min) & (df['easting'] <= este_max)]
asc = filter_area(asc)
desc = filter_area(desc)

# ==============================
# 2. Processamento (Melt, Interpolate, IDW, dV/dH)
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'], value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])
asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start=max(asc_long['date'].min(), desc_long['date'].min()), end=min(asc_long['date'].max(), desc_long['date'].max()), freq='MS')

def interpolate_ps(df, dates):
    if df.empty: return pd.DataFrame()
    dfs = []
    for (x, y), g in df.groupby(['easting','northing']):
        g = g.sort_values('date')
        interp = np.interp(pd.to_datetime(dates).astype(np.int64), g['date'].astype(np.int64), g['disp'])
        dfs.append(pd.DataFrame({'easting': x, 'northing': y, 'latitude': g['latitude'].iloc[0], 'longitude': g['longitude'].iloc[0], 'date': dates, 'disp': interp, 'incidence_angle': g['incidence_angle'].iloc[0], 'track_angle': g['track_angle'].iloc[0]}))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

if asc_interp.empty:
    print("ERRO: Sem dados após interpolação. Verifique coordenadas.")
    exit()

def idw(source, target, radius=150, power=2):
    out = []
    for d, src in source.groupby('date'):
        tgt = target[target['date']==d].copy()
        if src.empty or tgt.empty: continue
        tree = cKDTree(list(zip(src['easting'], src['northing'])))
        dist, idx = tree.query(list(zip(tgt['easting'], tgt['northing'])), k=5, distance_upper_bound=radius)
        vals, thetas = [], []
        for d_i, i_i in zip(dist, idx):
            m = np.isfinite(d_i)
            if not np.any(m): vals.append(np.nan); thetas.append(np.nan); continue
            w = 1/(d_i[m]**power)
            vals.append(np.sum(w*src.iloc[i_i[m]]['disp'])/np.sum(w))
            thetas.append(np.sum(w*src.iloc[i_i[m]]['incidence_angle'])/np.sum(w))
        tgt['disp_idw'] = vals; tgt['theta_desc'] = thetas
        out.append(tgt)
    return pd.concat(out, ignore_index=True)

asc_interp = idw(desc_interp, asc_interp).dropna(subset=['disp_idw'])

# Calcular Componente
orb_inc = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orb_inc) * np.cos(np.deg2rad(asc_interp['latitude'])))
def get_comps(row):
    ta, td, beta = np.deg2rad(row['incidence_angle']), np.deg2rad(row['theta_desc']), row['beta']
    denom = (np.cos(ta)*np.sin(td)*np.cos(beta) + np.cos(td)*np.sin(ta)*np.cos(beta))
    if denom == 0: return np.nan, np.nan
    dV = (row['disp_idw']*np.sin(ta)*np.cos(beta) + row['disp']*np.sin(td)*np.cos(beta))/denom
    dH = (row['disp_idw']*np.cos(ta) - row['disp']*np.cos(td))/denom
    return dV, dH
asc_interp[['dV', 'dH']] = asc_interp.apply(lambda x: pd.Series(get_comps(x)), axis=1)
asc_interp = asc_interp.dropna(subset=['dV', 'dH'])

# ==============================
# 3. Grelha e Agregação
# ==============================
xe = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+GRID_SIZE, GRID_SIZE)
ye = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+GRID_SIZE, GRID_SIZE)
xs, ys = xe - GRID_SIZE/2, ye - GRID_SIZE/2
asc_interp['cx'] = pd.cut(asc_interp['easting'], bins=xs, labels=False)
asc_interp['cy'] = pd.cut(asc_interp['northing'], bins=ys, labels=False)
asc_interp = asc_interp.dropna(subset=['cx','cy'])
asc_interp['cell_id'] = asc_interp['cx'].astype(int).astype(str)+"_"+asc_interp['cy'].astype(int).astype(str)

grid_data = [{'cell_id': f"{ix}_{iy}", 'geometry': box(xs[ix], ys[iy], xs[ix+1], ys[iy+1])} for ix in range(len(xs)-1) for iy in range(len(ys)-1)]
grid = gpd.GeoDataFrame(grid_data, crs='EPSG:3035')

# Agregação
agg = asc_interp.groupby(['cell_id','date']).agg(val=(TARGET_VAR,'mean')).reset_index()
agg.rename(columns={'val': TARGET_VAR}, inplace=True) # Nome correto da coluna

# ==============================
# 4. Recorte
# ==============================
points_gdf = gpd.GeoDataFrame(asc_interp, geometry=gpd.points_from_xy(asc_interp['easting'], asc_interp['northing']), crs="EPSG:3035").to_crs(epsg=3857)
grid_3857 = grid.to_crs(epsg=3857)
barragem_3857 = barragem.to_crs(epsg=3857)
grid_recort = gpd.overlay(grid_3857, barragem_3857, how='intersection')

poly = grid_recort.geometry.values; ids = grid_recort['cell_id'].values; tree = STRtree(poly)
valid_cells = set()
for pt in points_gdf.geometry:
    for idx in tree.query(pt):
        if poly[idx].contains(pt): valid_cells.add(ids[idx])

# Grelha final
grid_sel = grid_recort[grid_recort['cell_id'].isin(valid_cells)].copy()

# ==============================
# 5. PREPARAÇÃO PARA PLOT (SEM CLUSTERING)
# ==============================
# Atribuir IDs numéricos simples (1, 2, 3...) para identificar nos gráficos
grid_sel = grid_sel.sort_values('cell_id').reset_index(drop=True)
grid_sel['simple_id'] = range(1, len(grid_sel) + 1)

# Mapear cell_id original para simple_id
id_map = dict(zip(grid_sel['cell_id'], grid_sel['simple_id']))
agg = agg[agg['cell_id'].isin(valid_cells)].copy()
agg['simple_id'] = agg['cell_id'].map(id_map)

# Pivot para ter matriz (Index=SimpleID, Cols=Dates)
agg_pivot = agg.pivot(index='simple_id', columns='date', values=TARGET_VAR)

# Cores: Uma cor distinta para cada célula (usar cmap 'tab20' ou similar)
n_cells = len(grid_sel)
cmap = plt.get_cmap('tab20') if n_cells <= 20 else plt.get_cmap('gist_ncar')
colors = {i: cmap(i/n_cells) for i in range(n_cells)}

print(f"Total de Células Analisadas: {n_cells}")

# ==============================
# FIGURA 1: MAPA IDENTIFICADO
# ==============================
print("Gerando Mapa...")
fig1, ax1 = plt.subplots(figsize=(10, 10))
grid.to_crs(epsg=3857).boundary.plot(ax=ax1, color='white', lw=0.3, alpha=0.3)
grid_sel.boundary.plot(ax=ax1, color='black', lw=1)

# Plot das células com número
grid_sel.plot(ax=ax1, facecolor='none', edgecolor='black', linewidth=1.5)
for idx, row in grid_sel.iterrows():
    # Calcular centróide para o texto
    cent = row.geometry.centroid
    ax1.text(cent.x, cent.y, str(row['simple_id']), fontsize=12, color='red', fontweight='bold', ha='center', va='center')

ctx.add_basemap(ax1, source=ctx.providers.Esri.WorldImagery)
ax1.set_axis_off()
ax1.set_title(f"Localização das Células ({n_cells})", fontsize=14)
plt.show()

# ==============================
# FIGURA 2: SÉRIES TEMPORAIS (SUBPLOTS)
# ==============================
print("Gerando Séries...")
# Layout dinâmico: Tentar fazer uma grelha quadrada (ex: 4x4, 5x5)
cols = int(np.ceil(np.sqrt(n_cells)))
rows = int(np.ceil(n_cells / cols))

fig2, axes = plt.subplots(rows, cols, figsize=(4*cols, 3*rows), sharex=True, sharey=True)
axes = axes.flatten() # Achatar para iterar fácil

ymin, ymax = agg[TARGET_VAR].min(), agg[TARGET_VAR].max()
pad = (ymax - ymin) * 0.1

for i in range(len(axes)):
    ax = axes[i]
    if i < n_cells:
        sid = i + 1 # Simple ID é 1-based
        data = agg_pivot.loc[sid]
        
        ax.plot(data.index, data.values, color='black', lw=1.0)
        ax.set_title(f"Célula {sid}", fontsize=10, fontweight='bold', color='red')
        ax.set_ylim(ymin - pad, ymax + pad)
        #ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        
        if i % cols == 0: ax.set_ylabel(f'{TARGET_VAR} (mm)')
    else:
        ax.axis('off') # Esconder subplots vazios

plt.tight_layout()
plt.show()

# ==============================
# FIGURA 3: STL (COM ESCALA UNIFORME)
# ==============================
print("Gerando STL com escala uniforme...")

# 1. PRÉ-CÁLCULO: Calcular Decomposições e Limites Globais
decomps_storage = {}
vals_obs, vals_trend, vals_seas, vals_resid = [], [], [], []

for i in range(n_cells):
    sid = i + 1
    series = agg_pivot.loc[sid]
    try:
        # Calcular STL
        res = seasonal_decompose(pd.Series(series.values, index=series.index), 
                                 period=12, model='additive', extrapolate_trend='freq')
        decomps_storage[sid] = res
        
        # Guardar valores para cálculo de limites
        vals_obs.append(res.observed)
        vals_trend.append(res.trend)
        vals_seas.append(res.seasonal)
        vals_resid.append(res.resid)
    except:
        decomps_storage[sid] = None

# Função para calcular limites com margem de 10%
def get_global_limits(values_list):
    if not values_list: return (0, 1)
    # Concatenar todas as séries para achar min/max global
    all_vals = pd.concat(values_list)
    vmin, vmax = all_vals.min(), all_vals.max()
    margin = (vmax - vmin) * 0.1
    if margin == 0: margin = 0.1
    return (vmin - margin, vmax + margin)

# Calcular os limites globais
ylim_obs = get_global_limits(vals_obs)
ylim_trend = get_global_limits(vals_trend)
ylim_seas = get_global_limits(vals_seas)
ylim_resid = get_global_limits(vals_resid)

# 2. PLOTAGEM
fig3, axes = plt.subplots(n_cells, 4, figsize=(16, 1.0 * n_cells), sharex=True)
if n_cells == 1: axes = axes.reshape(1, 4)

for i in range(n_cells):
    sid = i + 1
    res = decomps_storage.get(sid)
    
    # Se a decomposição falhou, saltamos
    if res is None: continue

    # --- Observed ---
    ax = axes[i, 0]
    ax.plot(res.observed.index, res.observed, color='black', lw=1.0)
    ax.set_ylim(ylim_obs) # Escala Uniforme
    ax.set_ylabel(f'Célula {sid}', fontweight='bold', color='red', fontsize=9)
    if i == 0: ax.set_title("Observed (Original)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # --- Trend ---
    ax = axes[i, 1]
    ax.plot(res.trend.index, res.trend, color='blue', lw=1.0)
    ax.set_ylim(ylim_trend) # Escala Uniforme
    if i == 0: ax.set_title("Trend (Tendência)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # --- Seasonal ---
    ax = axes[i, 2]
    ax.plot(res.seasonal.index, res.seasonal, color='green', lw=1.0)
    ax.set_ylim(ylim_seas) # Escala Uniforme
    if i == 0: ax.set_title("Seasonal (Sazonalidade)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # --- Residual ---
    ax = axes[i, 3]
    ax.scatter(res.resid.index, res.resid, color='gray', s=5, alpha=0.7)
    ax.axhline(0, c='k', ls='--', lw=0.5)
    ax.set_ylim(ylim_resid) # Escala Uniforme
    if i == 0: ax.set_title("Residual (Ruído)")
    #ax.grid(True, linestyle=':', alpha=0.3)

    # Formatação X apenas na última linha
    if i == n_cells - 1:
        for ax_col in axes[i, :]: 
            #ax_col.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
            ax_col.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# CARREGAMENTO DOS DADOS EXTERNOS (Nível, Temp, Prec)
# ==============================

# 1. Nível da Albufeira
try:
    df_nivel_raw = pd.read_excel("data/alqueva_nivel.xlsx") # Ajusta o caminho se necessário
    df_nivel_raw['data'] = pd.to_datetime(df_nivel_raw['data'])
    # Resample mensal para bater certo com a InSAR
    df_nivel = df_nivel_raw.set_index('data').resample('MS').mean().reset_index()
    df_nivel.rename(columns={'nivel': 'nivel_smooth'}, inplace=True) 
except:
    print("Aviso: Ficheiro de nível não encontrado. A criar dados fictícios para teste.")
    d_rng = pd.date_range(start='2019-01-01', end='2023-12-01', freq='MS')
    df_nivel = pd.DataFrame({'data': d_rng, 'nivel_smooth': 140 + 10*np.sin(np.arange(len(d_rng))/6)})

# 2. Temperatura
try:
    df_temp_raw = pd.read_excel("data/alqueva_temp.xlsx")
    df_temp_raw['data'] = pd.to_datetime(df_temp_raw['data'])
    df_temp = df_temp_raw.set_index('data').resample('MS').mean().reset_index()
    df_temp.rename(columns={'med': 'med_smooth'}, inplace=True)
except:
    df_temp = pd.DataFrame({'data': df_nivel['data'], 'med_smooth': 15 + 8*np.cos(np.arange(len(df_nivel))/6)})

# 3. Precipitação
try:
    df_prec = pd.read_excel("data/prec.xlsx")
    df_prec['data'] = pd.to_datetime(df_prec['data'])
except:
    df_prec = pd.DataFrame({'data': df_nivel['data'], 'prec': np.random.uniform(0, 100, len(df_nivel))})
    df_prec['prec_acum_anual'] = df_prec['prec'].cumsum() # Apenas para o gráfico não dar erro

In [ ]:
# ==============================================================================
# SCRIPT FINAL UNIFICADO: ANÁLISE IN SAR + MODELO HST + VALIDAÇÃO + RESUMO
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. CONFIGURAÇÃO INICIAL
# ------------------------------------------------------------------------------
#PONTOS_A_ANALISAR = [49, 58, 59, 69, 23]
PONTOS_A_ANALISAR = [69]  
summary_list = []

# Função auxiliar para métricas
def calc_metrics(real, pred):
    r2 = r2_score(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    mae = mean_absolute_error(real, pred)
    return r2, rmse, mae

# ==============================================================================
# LOOP DE ANÁLISE POR CÉLULA
# ==============================================================================
for sid in PONTOS_A_ANALISAR:
    if sid not in agg_pivot.index:
        print(f"Célula {sid} não encontrada.")
        continue

    # --- A. PREPARAÇÃO E ALINHAMENTO ---
    y_series = agg_pivot.loc[sid].dropna()
    dates = y_series.index
    
    # Alinhamento das variáveis externas (Nível e Temperatura)
    h = df_nivel.set_index('data')['nivel_smooth'].reindex(dates).interpolate().values
    temp = df_temp.set_index('data')['med_smooth'].reindex(dates).interpolate().values
    
    t_days = (dates - dates.min()).days.values
    s = 2 * np.pi * t_days / 365.25

    # --- B. CONSTRUÇÃO DA MATRIZ X (Modelo HST Robusto) ---
    # F1: Hidrostático (Grau 2) | F2: Sazonal (Ciclo Anual) | F3: Tempo (Linear + Log)
    F1 = np.column_stack([h, h**2]) 
    F2 = np.column_stack([np.cos(s), np.sin(s)])
    F3 = np.column_stack([t_days, np.log(t_days + 1)])
    X = np.hstack([F1, F2, F3])

    # --- C. FIGURA 1: VARIÁVEIS EXTERNAS (Compacta) ---
    fig1, axs1 = plt.subplots(3, 1, figsize=(12, 4), sharex=True)
    fig1.suptitle(f'Célula {sid}: Variáveis Externas', fontsize=11, fontweight='bold')
    
    axs1[0].plot(dates, y_series.values, 'k-o', markersize=2, lw=0.7, label="InSAR")
    axs1[0].set_ylabel("dV (mm)", fontsize=8)
    
    axs1[1].plot(df_nivel['data'], df_nivel['nivel_smooth'], color='navy', lw=1)
    axs1[1].set_ylabel("Nível (m)", fontsize=8)
    
    axs1[2].plot(df_temp['data'], df_temp['med_smooth'], color='darkred', lw=1)
    axs1[2].set_ylabel("Temp (°C)", fontsize=8)
    
    for ax in axs1: 
        ax.grid(True, alpha=0.15)
        ax.tick_params(labelsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # --- D. FIGURA 2: AJUSTE GLOBAL E MÉTRICAS ---
    model_all = LinearRegression().fit(X, y_series.values)
    y_all_pred = model_all.predict(X)
    r2_all, rmse_all, mae_all = calc_metrics(y_series.values, y_all_pred)

    plt.figure(figsize=(12, 3))
    plt.plot(dates, y_series.values, 'k.', alpha=0.3, label="Observed")
    plt.plot(dates, y_all_pred, color='darkgreen', lw=1.5, label="Global HST Fit")
    
    # Caixa de texto com métricas globais
    txt_all = f"AJUSTE GLOBAL:\nR²: {r2_all:.3f} | RMSE: {rmse_all:.3f}mm"
    plt.gca().text(0.02, 0.92, txt_all, transform=plt.gca().transAxes, fontsize=9, 
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='green', alpha=0.05))
    
    plt.title(f"Ajuste Total do Modelo HST - Célula {sid}", fontsize=11, fontweight='bold')
    plt.ylabel("dV (mm)", fontsize=8)
    plt.grid(True, alpha=0.15)
    plt.tick_params(labelsize=8)
    plt.tight_layout()
    plt.show()

    # --- E. FIGURA 3: VALIDAÇÃO TREINO/TESTE E RESÍDUOS ---
    X_train, X_test, y_train, y_test = train_test_split(X, y_series.values, test_size=0.3, shuffle=False)
    model_val = LinearRegression().fit(X_train, y_train)
    
    y_tr_p = model_val.predict(X_train)
    y_te_p = model_val.predict(X_test)
    tr_dates, te_dates = dates[:len(y_train)], dates[len(y_train):]
    
    # Métricas Treino/Teste
    r2_tr, rmse_tr, mae_tr = calc_metrics(y_train, y_tr_p)
    r2_te, rmse_te, mae_te = calc_metrics(y_test, y_te_p)

    fig3, axs3 = plt.subplots(2, 1, figsize=(12, 5), sharex=True, gridspec_kw={'height_ratios': [2.5, 1]})
    axs3[0].plot(dates, y_series.values, color='black', lw=1, alpha=0.2)
    axs3[0].plot(tr_dates, y_tr_p, 'b--', lw=1.2, label="Treino")
    axs3[0].plot(te_dates, y_te_p, 'r:', lw=1.8, label="Teste (Previsão)")
    axs3[0].axvline(te_dates[0], color='grey', ls='--', alpha=0.5)
    
    # Inserção das métricas no Plot de Validação
    axs3[0].text(0.02, 0.05, f"TREINO: R² {r2_tr:.3f} | RMSE {rmse_tr:.3f}mm", transform=axs3[0].transAxes, 
                 fontsize=8, color='blue', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axs3[0].text(0.98, 0.05, f"TESTE: R² {r2_te:.3f} | RMSE {rmse_te:.3f}mm", transform=axs3[0].transAxes, 
                 fontsize=8, color='red', ha='right', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # Resíduos
    axs3[1].bar(tr_dates, y_train - y_tr_p, color='blue', alpha=0.3)
    axs3[1].bar(te_dates, y_test - y_te_p, color='red', alpha=0.3)
    axs3[1].axhline(0, color='black', lw=0.5)
    
    fig3.suptitle(f'Validação Treino/Teste - Célula {sid}', fontsize=11, fontweight='bold')
    for ax in axs3: ax.grid(True, alpha=0.1); ax.tick_params(labelsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # --- F. GUARDAR NO RESUMO ---
    summary_list.append({
        'Célula': sid,
        'R² Global': r2_all,
        'RMSE Global': rmse_all,
        'R² Treino': r2_tr,
        'RMSE Treino': rmse_tr,
        'R² Teste': r2_te,
        'RMSE Teste': rmse_te
    })

# ==============================================================================
# GERAÇÃO DO RELATÓRIO FINAL (FORA DO LOOP)
# ==============================================================================
summary_df = pd.DataFrame(summary_list)

print("\n" + "="*90)
print("📈 RESUMO FINAL DE PERFORMANCE - COMPARAÇÃO ENTRE CÉLULAS")
print("="*90)

pd.options.display.float_format = '{:,.4f}'.format
print(summary_df.to_string(index=False))

if not summary_df.empty:
    melhor = summary_df.loc[summary_df['R² Teste'].idxmax()]
    print("\n" + "-"*90)
    print(f"✅ MELHOR MODELO (PREVISÃO): Célula {melhor['Célula']:.0f}")
    print(f"   R² Teste: {melhor['R² Teste']:.4f} | RMSE Teste: {melhor['RMSE Teste']:.4f} mm")
    print("-" * 90)

In [ ]:
# ==============================================================================
# SCRIPT: DIAGNÓSTICO DE ESTABILIDADE (COMPARAÇÃO R² TREINO VS TESTE)
# ==============================================================================

# 1. PREPARAÇÃO DOS DADOS
# Utilizamos o summary_df gerado no passo anterior
df_plot = summary_df.copy()
df_plot['Célula'] = df_plot['Célula'].astype(str)

# 2. CONFIGURAÇÃO DO GRÁFICO
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(df_plot['Célula']))
width = 0.35  # Largura das barras

# Criar as barras
rects1 = ax.bar(x - width/2, df_plot['R² Treino'], width, label='R² Treino (Calibração)', 
                alpha=0.8, edgecolor='white', lw=0.5)
rects2 = ax.bar(x + width/2, df_plot['R² Teste'], width, label='R² Teste (Previsão)', 
                alpha=0.8, edgecolor='white', lw=0.5)

# 3. DESTAQUE VISUAL PARA PERFORMANCE NEGATIVA
# Se o R² de Teste for negativo, adicionamos uma marca de alerta
for i, r2_test in enumerate(df_plot['R² Teste']):
    if r2_test < 0:
        ax.annotate('⚠️ FALHA', xy=(i + width/2, 0), xytext=(0, -15),
                    textcoords="offset points", ha='center', va='top',
                    color=C_TEST, fontweight='bold', fontsize=8)

# 4. ESTILIZAÇÃO "ELITE"
ax.set_title('ESTABILIDADE DO MODELO HST POR CÉLULA (COROAMENTO)', fontsize=11, fontweight='bold', loc='left', pad=20)
ax.set_ylabel('Coeficiente de Determinação ($R^2$)')
ax.set_xticks(x)
ax.set_xticklabels(df_plot['Célula'])
ax.axhline(0, color='#2d3436', lw=0.8) # Linha do zero bem definida
ax.set_ylim(min(df_plot['R² Teste'].min() - 0.2, -0.5), 1.1) # Ajuste dinâmico do eixo Y

# Adicionar grelha apenas horizontal para facilitar a leitura dos valores
ax.yaxis.grid(True, alpha=0.3)
ax.xaxis.grid(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legenda elegante
ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=2)

# Adicionar os valores por cima das barras (apenas se forem positivos e significativos)
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        if height > 0.1:
            ax.annotate(f'{height:.2f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=7, color='#636e72')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

Google Earth Pro

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import fiona
import contextily as ctx
from shapely.geometry import Point
from scipy.spatial import cKDTree
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# ==============================================================================
# 1. CONFIGURAÇÕES
# ==============================================================================
TARGET_VAR = 'dV'
PATH_ASC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
PATH_DESC = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
PATH_KML = 'data/Blocos_Alqueva.kml'
PATH_NIVEL = "data/alqueva_nivel.xlsx"
PATH_TEMP = "data/alqueva_temp.xlsx"

# Alarguei um pouco os limites para garantir que não cortamos nada por erro
norte_min, norte_max = 1854000, 1856500
este_min, este_max = 2791500, 2794000

# ==============================================================================
# 2. CARREGAMENTO E FILTRAGEM (COM VERIFICAÇÃO)
# ==============================================================================
print("--- INICIANDO DIAGNÓSTICO ---")
asc_raw = pd.read_csv(PATH_ASC)
desc_raw = pd.read_csv(PATH_DESC)
print(f"Total linhas CSV Asc: {len(asc_raw)}")

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) & 
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc_raw)
desc = filter_area(desc_raw)
print(f"Após filtro geográfico: Asc={len(asc)}, Desc={len(desc)}")

if len(asc) == 0:
    print("ERRO: O filtro de coordenadas eliminou todos os pontos. Verifique norte_min/este_min.")

# ==============================================================================
# 3. INTERPOLAÇÃO E COMBINAÇÃO (IDW)
# ==============================================================================
def melt_to_long(df):
    disp_cols = df.columns[24:]
    long_df = df.melt(id_vars=['easting','northing','incidence_angle','latitude','longitude'], 
                      value_vars=disp_cols, var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'])
    return long_df

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)
common_dates = pd.date_range(start='2019-01-01', end='2023-12-01', freq='MS')

def interpolate_dates(df, dates):
    dfs_list = []
    for (x, y), group in df.groupby(['easting', 'northing']):
        group = group.sort_values('date')
        interp_vals = np.interp(dates.astype(np.int64), group['date'].astype(np.int64), group['disp'])
        dfs_list.append(pd.DataFrame({
            'easting': x, 'northing': y, 'date': dates, 'disp': interp_vals,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'latitude': group['latitude'].iloc[0], 'longitude': group['longitude'].iloc[0]
        }))
    return pd.concat(dfs_list) if dfs_list else pd.DataFrame()

asc_interp = interpolate_dates(asc_long, common_dates)
desc_interp = interpolate_dates(desc_long, common_dates)

def combine_orbits(asc_df, desc_df):
    if asc_df.empty or desc_df.empty: return pd.DataFrame()
    combined_list = []
    for date, group_asc in asc_df.groupby('date'):
        group_desc = desc_df[desc_df['date'] == date]
        if group_desc.empty: continue
        tree = cKDTree(group_desc[['easting', 'northing']].values)
        dist, idx = tree.query(group_asc[['easting', 'northing']].values, k=5, distance_upper_bound=100)
        mask = np.all(dist < 100, axis=1)
        if not np.any(mask): continue
        valid_asc = group_asc[mask].copy()
        w = 1 / (dist[mask]**2 + 1e-6)
        valid_asc['disp_desc'] = np.sum(w * group_desc['disp'].values[idx[mask]], axis=1) / np.sum(w, axis=1)
        valid_asc['theta_desc'] = np.sum(w * group_desc['incidence_angle'].values[idx[mask]], axis=1) / np.sum(w, axis=1)
        combined_list.append(valid_asc)
    return pd.concat(combined_list) if combined_list else pd.DataFrame()

df_combined = combine_orbits(asc_interp, desc_interp)
print(f"Pontos após combinação dV: {len(df_combined)}")

# Cálculo dV
theta_a = np.deg2rad(df_combined['incidence_angle'])
theta_d = np.deg2rad(df_combined['theta_desc'])
df_combined['dV'] = (df_combined['disp'] * np.sin(theta_d) - df_combined['disp_desc'] * np.sin(theta_a)) / np.sin(theta_a + theta_d)

# ==============================================================================
# 4. O FILTRO CRÍTICO (KML + JOIN)
# ==============================================================================
# 1. Converter InSAR para GeoDataFrame
gdf_insar = gpd.GeoDataFrame(df_combined, 
    geometry=gpd.points_from_xy(df_combined['longitude'], df_combined['latitude']), 
    crs="EPSG:4326").to_crs(epsg=3763)

# 2. Ler KML e aplicar BUFFER
fiona.drvsupport.supported_drivers['KML'] = 'rw'
blocos_gdf = gpd.read_file(PATH_KML, driver='KML').to_crs(epsg=3763)

# --- AQUI ESTÁ A CORREÇÃO ---
# Criamos uma margem de 15 metros à volta do teu desenho para garantir que apanhamos os pontos
blocos_gdf['geometry'] = blocos_gdf.geometry.buffer(15) 

pontos_filtrados = gpd.sjoin(gdf_insar, blocos_gdf, how="inner", predicate="within")
print(f"Pontos INSAR dentro dos polígonos: {len(pontos_filtrados)}")

if len(pontos_filtrados) == 0:
    print("AVISO: O Join falhou. Verifique se o KML está na zona correta da barragem.")
    # Vamos criar um gráfico de diagnóstico para ver onde estão os pontos e o KML
    fig, ax = plt.subplots()
    gdf_insar.plot(ax=ax, color='blue', markersize=1, label='Pontos InSAR')
    blocos_gdf.plot(ax=ax, color='red', alpha=0.5, label='Teu KML')
    plt.legend(); plt.show()

# ==============================================================================
# 5. RESTO DO MODELO (Só corre se houver dados)
# ==============================================================================
if not pontos_filtrados.empty:
    agg = pontos_filtrados.groupby(['Name', 'date']).agg({'dV': 'mean'}).reset_index()
    agg_pivot = agg.pivot(index='Name', columns='date', values='dV')

    df_nivel = pd.read_excel(PATH_NIVEL); df_nivel['data'] = pd.to_datetime(df_nivel['data'])
    df_temp = pd.read_excel(PATH_TEMP); df_temp['data'] = pd.to_datetime(df_temp['data'])

    summary = []
    for bloco_nome in agg_pivot.index:
        y_vals = agg_pivot.loc[bloco_nome]
        h = df_nivel.set_index('data')['nivel'].reindex(y_vals.index).interpolate().values
        temp = df_temp.set_index('data')['med'].reindex(y_vals.index).interpolate().values
        
        valid_mask = ~np.isnan(y_vals.values) & ~np.isnan(h) & ~np.isnan(temp)
        if valid_mask.sum() < 10: continue
        
        y, h_val, t_val = y_vals.values[valid_mask], h[valid_mask], temp[valid_mask]
        t_days = (y_vals.index[valid_mask] - y_vals.index[valid_mask].min()).days.values
        s = 2 * np.pi * t_days / 365.25
        
        X = np.column_stack([h_val, h_val**2, np.cos(s), np.sin(s), t_days, np.log(t_days+1)])
        reg = LinearRegression().fit(X, y)
        summary.append({'Bloco': bloco_nome, 'R2': r2_score(y, reg.predict(X))})

    print("\n--- RESULTADOS FINAIS ---")
    print(pd.DataFrame(summary))
else:
    print("Script terminado sem dados para processar.")